# NDJF Manual Verification and Position QA

This notebook adds peak-position diagnostics, simple candidate intensity metrics, a manual-review scaffold, and batch quicklook plots for visual event verification.

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

import pandas as pd

from jpcz_catalog.analysis import (
    add_candidate_intensity_metrics,
    add_position_group_columns,
    annotate_events_with_peak_position,
    build_manual_verification_scaffold,
)
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.verification import render_manual_verification_summary, write_text_summary

CATALOG_PATH = Path("outputs/verification/jpcz_catalog_ndjf.csv")
AUTO_CATALOG_PATH = Path("outputs/verification/jpcz_catalog_ndjf_position_intensity.csv")
SCAFFOLD_PATH = Path("outputs/verification/jpcz_catalog_ndjf_manual_verification.csv")
SUMMARY_PATH = Path("outputs/verification/ndjf_manual_verification_summary.md")
PLOT_DIR = Path("outputs/verification/ndjf_event_quicklooks")
CLOUD_VARIABLE = None

catalog_df = pd.read_csv(
    CATALOG_PATH,
    parse_dates=["event_start", "event_end", "event_peak"],
)
ds = open_arco_era5()

if CLOUD_VARIABLE is not None and CLOUD_VARIABLE not in ds.data_vars:
    print(f"Requested cloud variable '{CLOUD_VARIABLE}' is not available in this ERA5 store. Falling back to wind + convergence only.")
    CLOUD_VARIABLE = None

augmented_catalog = annotate_events_with_peak_position(ds, catalog_df)
augmented_catalog = add_candidate_intensity_metrics(augmented_catalog)
augmented_catalog = add_position_group_columns(augmented_catalog)
augmented_catalog.to_csv(AUTO_CATALOG_PATH, index=False)

verification_scaffold = build_manual_verification_scaffold(augmented_catalog)
verification_scaffold.to_csv(SCAFFOLD_PATH, index=False)

summary_text = render_manual_verification_summary(
    total_events=len(verification_scaffold),
    auto_catalog_name=AUTO_CATALOG_PATH.name,
    scaffold_name=SCAFFOLD_PATH.name,
    plot_dir_name=PLOT_DIR.name,
    cloud_variable=CLOUD_VARIABLE,
)
summary_path = write_text_summary(SUMMARY_PATH, summary_text)

if PERSIST_OUTPUTS_TO_DRIVE:
    for path in [AUTO_CATALOG_PATH, SCAFFOLD_PATH, summary_path]:
        shutil.copy2(path, Path(DRIVE_OUTPUT_DIR) / path.name)

print(summary_text)
verification_scaffold.head()


In [ ]:
VARIABLE_KEYWORDS = (
    "cloud",
    "olr",
    "outgoing",
    "radiation",
    "longwave",
    "shortwave",
    "toa",
    "top_net",
    "temperature",
)

candidate_variables = sorted(
    name for name in ds.data_vars
    if any(keyword in name.lower() for keyword in VARIABLE_KEYWORDS)
)

print(f"Total variables in ARCO ERA5 store: {len(ds.data_vars)}")
print("Candidate cloud/OLR-like variables:")
for name in candidate_variables:
    print(" -", name)

if CLOUD_VARIABLE is None:
    print("\nCLOUD_VARIABLE is currently None. If you want a cloud proxy pass, set CLOUD_VARIABLE in cell 2, then rerun cells 2-6.")
else:
    print(f"\nCLOUD_VARIABLE is currently set to: {CLOUD_VARIABLE}")


In [ ]:
from pathlib import Path
import shutil

import pandas as pd

from jpcz_catalog.analysis import load_event_peak_snapshot
from jpcz_catalog.config import JPCZ_POLYGON_VERTICES, VORTICITY_BOX, WORKING_DOMAIN
from jpcz_catalog.detect import compute_divergence_field, prepare_detection_geometry
from jpcz_catalog.plotting import plot_event_peak_quicklook

PLOT_START = 0
PLOT_STOP = 12
OVERWRITE_EXISTING_PLOTS = False

if "verification_scaffold" not in globals():
    verification_scaffold = pd.read_csv(
        SCAFFOLD_PATH,
        parse_dates=["event_start", "event_end", "event_peak"],
    )
if "ds" not in globals():
    ds = open_arco_era5()

PLOT_DIR.mkdir(parents=True, exist_ok=True)
drive_plot_dir = Path(DRIVE_OUTPUT_DIR) / "ndjf_event_quicklooks"
if PERSIST_OUTPUTS_TO_DRIVE:
    drive_plot_dir.mkdir(parents=True, exist_ok=True)

selected_events = verification_scaffold.iloc[PLOT_START:PLOT_STOP].copy()
geometry = None

for idx, row in selected_events.iterrows():
    peak_snapshot = load_event_peak_snapshot(
        ds,
        row["event_peak"],
        domain=WORKING_DOMAIN,
        cloud_variable=CLOUD_VARIABLE,
    )

    if geometry is None:
        geometry = prepare_detection_geometry(
            peak_snapshot.longitude,
            peak_snapshot.latitude,
            JPCZ_POLYGON_VERTICES,
        )

    divergence_field = compute_divergence_field(
        peak_snapshot,
        dx=geometry.dx,
        dy=geometry.dy,
    )

    cloud_field = peak_snapshot[CLOUD_VARIABLE] if CLOUD_VARIABLE is not None else None
    out_name = f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{idx:03d}.png"
    out_path = PLOT_DIR / out_name
    if out_path.exists() and not OVERWRITE_EXISTING_PLOTS:
        print(f"Skipping existing plot: {out_name}")
        continue

    plot_event_peak_quicklook(
        peak_snapshot,
        divergence_field,
        domain=WORKING_DOMAIN,
        polygon_vertices=JPCZ_POLYGON_VERTICES,
        vorticity_box=VORTICITY_BOX,
        title=(
            f"NDJF event {idx} | peak {pd.Timestamp(row['event_peak']).strftime('%Y-%m-%d %H:%M UTC')}\n"
            f"peak D={row['event_peak_D_1e5_s-1']:.2f} | duration={int(row['duration_hours'])} h | auto lat group={row['lat_position_group_auto']}"
        ),
        cloud_field=cloud_field,
        cloud_label=CLOUD_VARIABLE or "Cloud proxy",
        max_location=(row["peak_max_convergence_lat"], row["peak_max_convergence_lon"]),
        centroid_location=(row["peak_convergence_centroid_lat"], row["peak_convergence_centroid_lon"]),
        save_path=out_path,
    )

    if PERSIST_OUTPUTS_TO_DRIVE:
        shutil.copy2(out_path, drive_plot_dir / out_path.name)

    print(f"Saved {out_path}")


In [ ]:
preview_columns = [
    "event_peak",
    "event_peak_D_1e5_s-1",
    "duration_hours",
    "peak_max_convergence_lat",
    "peak_max_convergence_lon",
    "peak_convergence_centroid_lat",
    "peak_convergence_centroid_lon",
    "lat_position_group_auto",
    "lon_position_group_auto",
    "candidate_peak_convergence_1e5_s-1",
    "candidate_duration_weighted_convergence",
    "candidate_positive_box_vorticity_1e5_s-1",
    "candidate_peak_duration_vorticity_index",
    "verified_event",
    "cloud_band_present",
    "position_group_manual",
    "satellite_checked",
    "satellite_cloud_band_match",
    "satellite_source",
]

verification_scaffold.loc[PLOT_START:PLOT_STOP - 1, preview_columns]


In [ ]:
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, Image, clear_output, display

if "verification_scaffold" not in globals():
    verification_scaffold = pd.read_csv(
        SCAFFOLD_PATH,
        parse_dates=["event_start", "event_end", "event_peak"],
    )

local_plot_dir = PLOT_DIR
drive_plot_dir = Path(DRIVE_OUTPUT_DIR) / "ndjf_event_quicklooks"
drive_scaffold_path = Path(DRIVE_OUTPUT_DIR) / SCAFFOLD_PATH.name

def quicklook_filename(row_index: int, row: pd.Series) -> str:
    timestamp = pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')
    return f"{timestamp}_idx{row_index:03d}.png"

def quicklook_path(row_index: int, row: pd.Series) -> Path:
    name = quicklook_filename(row_index, row)
    local_path = local_plot_dir / name
    if local_path.exists():
        return local_path
    drive_path = drive_plot_dir / name
    return drive_path

current_index = widgets.BoundedIntText(value=0, min=0, max=len(verification_scaffold) - 1, description='Row')
verified_widget = widgets.Dropdown(
    options=['', 'yes', 'no', 'uncertain'],
    description='Verified',
)
cloud_widget = widgets.Dropdown(
    options=['', 'yes', 'no', 'uncertain'],
    description='Cloud proxy',
)
position_widget = widgets.Dropdown(
    options=['', 'north-shifted', 'central', 'south-shifted', 'west-shifted', 'east-shifted', 'complex'],
    description='Manual pos',
)
manual_lat_widget = widgets.Text(description='Manual lat')
manual_lon_widget = widgets.Text(description='Manual lon')
satellite_checked_widget = widgets.Dropdown(
    options=['', 'planned', 'yes', 'no'],
    description='Satellite',
)
satellite_match_widget = widgets.Dropdown(
    options=['', 'yes', 'no', 'uncertain'],
    description='Sat match',
)
satellite_source_widget = widgets.Text(description='Sat source')
forcing_widget = widgets.Text(description='Jet note')
notes_widget = widgets.Textarea(description='Notes', layout=widgets.Layout(width='100%', height='120px'))
satellite_notes_widget = widgets.Textarea(description='Sat notes', layout=widgets.Layout(width='100%', height='90px'))
save_button = widgets.Button(description='Save row', button_style='success')
next_button = widgets.Button(description='Save + next', button_style='primary')
prev_button = widgets.Button(description='Prev row')
jump_unreviewed_button = widgets.Button(description='Next blank')
status_html = widgets.HTML()
meta_html = widgets.HTML()
image_out = widgets.Output()

def normalize_float(value):
    if value is None or value == '':
        return pd.NA
    try:
        if pd.isna(value):
            return pd.NA
    except TypeError:
        pass
    return float(value)

def save_current(_=None):
    idx = int(current_index.value)
    verification_scaffold.at[idx, 'verified_event'] = verified_widget.value
    verification_scaffold.at[idx, 'cloud_band_present'] = cloud_widget.value
    verification_scaffold.at[idx, 'position_group_manual'] = position_widget.value
    verification_scaffold.at[idx, 'manual_peak_convergence_lat'] = normalize_float(manual_lat_widget.value)
    verification_scaffold.at[idx, 'manual_peak_convergence_lon'] = normalize_float(manual_lon_widget.value)
    verification_scaffold.at[idx, 'satellite_checked'] = satellite_checked_widget.value
    verification_scaffold.at[idx, 'satellite_cloud_band_match'] = satellite_match_widget.value
    verification_scaffold.at[idx, 'satellite_source'] = satellite_source_widget.value
    verification_scaffold.at[idx, 'satellite_notes'] = satellite_notes_widget.value
    verification_scaffold.at[idx, 'upper_level_forcing_note'] = forcing_widget.value
    verification_scaffold.at[idx, 'verification_notes'] = notes_widget.value
    verification_scaffold.to_csv(SCAFFOLD_PATH, index=False)
    if PERSIST_OUTPUTS_TO_DRIVE:
        verification_scaffold.to_csv(drive_scaffold_path, index=False)
    status_html.value = f"<b>Saved row {idx}</b> to {SCAFFOLD_PATH.name}"

def load_row(idx: int):
    row = verification_scaffold.iloc[idx]
    verified_widget.value = row.get('verified_event', '') if pd.notna(row.get('verified_event', '')) else ''
    cloud_widget.value = row.get('cloud_band_present', '') if pd.notna(row.get('cloud_band_present', '')) else ''
    position_widget.value = row.get('position_group_manual', '') if pd.notna(row.get('position_group_manual', '')) else ''
    manual_lat_widget.value = str(float(row['manual_peak_convergence_lat'])) if pd.notna(row.get('manual_peak_convergence_lat')) else ''
    manual_lon_widget.value = str(float(row['manual_peak_convergence_lon'])) if pd.notna(row.get('manual_peak_convergence_lon')) else ''
    satellite_checked_widget.value = row.get('satellite_checked', '') if pd.notna(row.get('satellite_checked', '')) else ''
    satellite_match_widget.value = row.get('satellite_cloud_band_match', '') if pd.notna(row.get('satellite_cloud_band_match', '')) else ''
    satellite_source_widget.value = row.get('satellite_source', '') if pd.notna(row.get('satellite_source', '')) else ''
    satellite_notes_widget.value = row.get('satellite_notes', '') if pd.notna(row.get('satellite_notes', '')) else ''
    forcing_widget.value = row.get('upper_level_forcing_note', '') if pd.notna(row.get('upper_level_forcing_note', '')) else ''
    notes_widget.value = row.get('verification_notes', '') if pd.notna(row.get('verification_notes', '')) else ''

    plot_path = quicklook_path(idx, row)
    meta_html.value = (
        f"<b>event_peak:</b> {pd.Timestamp(row['event_peak'])} &nbsp; &nbsp; "
        f"<b>month:</b> {row['month']} &nbsp; &nbsp; "
        f"<b>peak D:</b> {row['event_peak_D_1e5_s-1']:.2f} &nbsp; &nbsp; "
        f"<b>duration:</b> {int(row['duration_hours'])} h<br>"
        f"<b>auto max lat/lon:</b> {row['peak_max_convergence_lat']:.2f}, {row['peak_max_convergence_lon']:.2f} &nbsp; &nbsp; "
        f"<b>auto centroid lat/lon:</b> {row['peak_convergence_centroid_lat']:.2f}, {row['peak_convergence_centroid_lon']:.2f}<br>"
        f"<b>auto lat group:</b> {row['lat_position_group_auto']} &nbsp; &nbsp; "
        f"<b>auto lon group:</b> {row['lon_position_group_auto']}<br>"
        f"<b>CLOUD_VARIABLE:</b> {CLOUD_VARIABLE or 'None'}"
    )

    with image_out:
        clear_output(wait=True)
        if plot_path.exists():
            display(Image(filename=str(plot_path), width=950))
        else:
            display(HTML(f"<b>Missing quicklook:</b> {plot_path}"))

def on_row_change(change):
    if change['name'] == 'value':
        load_row(int(change['new']))

def save_and_next(_):
    save_current()
    if current_index.value < current_index.max:
        current_index.value += 1

def go_prev(_):
    if current_index.value > current_index.min:
        current_index.value -= 1

def jump_next_blank(_):
    blanks = verification_scaffold.index[verification_scaffold['verified_event'].fillna('') == '']
    blanks = [i for i in blanks if i > current_index.value]
    if blanks:
        current_index.value = blanks[0]
    else:
        status_html.value = '<b>No later blank rows found.</b>'

current_index.observe(on_row_change)
save_button.on_click(save_current)
next_button.on_click(save_and_next)
prev_button.on_click(go_prev)
jump_unreviewed_button.on_click(jump_next_blank)

controls = widgets.VBox([
    widgets.HBox([current_index, prev_button, save_button, next_button, jump_unreviewed_button]),
    status_html,
    meta_html,
    widgets.HBox([verified_widget, cloud_widget, position_widget]),
    widgets.HBox([manual_lat_widget, manual_lon_widget]),
    widgets.HBox([satellite_checked_widget, satellite_match_widget]),
    satellite_source_widget,
    forcing_widget,
    notes_widget,
    satellite_notes_widget,
])

display(controls)
display(image_out)
load_row(int(current_index.value))
